# Parse Minnie Clustering — Step-by-Step Workflow

This notebook walks through **how this repository uses LinkML schemas and auto-generated Pydantic models** to build and write connectivity/clustering data. It uses the **same Minnie dataset** as `parse_minnie_clustering.ipynb` (CAVE minnie65_phase3_v1 v1412, parquet feature/connectivity tables, minnie_cell_features.csv) with step-by-step markdown comments.

**Pipeline overview:**
1. **Schemas** are defined in LinkML (YAML under `schemas/`).
2. **Pydantic models** are generated from those schemas (e.g. via `scripts/generate_models.sh`).
3. You **instantiate** those models with your data (e.g. `DataSet`, `DataItem`, `Cluster`).
4. **Arrow utilities** turn model instances into PyArrow tables: `build_arrow_schema()`, `models_to_table()`, `attach_linkml_metadata()`.
5. Tables are written to **Delta Lake** with `write_deltalake()` for storage and querying.


In [40]:
# Imports: package models (from LinkML), Arrow/Delta helpers, and Minnie data dependencies
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake
import caveclient
import standard_transform

# Add repo root so we can import the generated package (run from repo root or code/)
_repo = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(_repo / "src"))

from connects_common_connectivity.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Unit,
    CellFeatureDefinition,
    CellFeatureSet,
    Cluster,
    ClusterMembership,
    CellCellConnectivityLong,
    SynapticMeasurementType,
)

---

## 1. Where do the models come from?

The classes (`DataSet`, `DataItem`, `Cluster`, etc.) are **generated from LinkML YAML schemas** in `schemas/`. The script `scripts/generate_models.sh` (typically using `linkml-codegen` or similar) produces Python Pydantic models in `src/connects_common_connectivity/models.py`. When the YAML changes, you re-run the generator to refresh the models.

## 2a. Loading the Minnie dataset (CAVE + feature table)

The original pipeline uses **CAVE** (minnie65_phase3_v1, materialization version 1412) for nucleus metadata and a precomputed **feature parquet** for cells that have clustering features. Here we load: (1) `nucleus_detection_lookup_v1` → `nuc_df` (filter to `pt_root_id != 0`); (2) the feature table → `dfm` (one row per cell with morphology/connectivity features). Associations between DataItems and the DataSet are built from `dfm`, so only cells with features are linked to this dataset.

In [41]:
# CAVE client and nucleus table (Minnie65 v1412)
client = caveclient.CAVEclient("minnie65_phase3_v1", auth_token=os.environ["CUSTOM_KEY"])
version = 1412
client.materialize.version = version
nuc_df = client.materialize.query_view("nucleus_detection_lookup_v1")
nuc_df.query("pt_root_id!=0", inplace=True)

# Feature table: one row per cell with computed features (from joint clustering pipeline)
dfm = pd.read_parquet("../data/minnie1412/minnie_features.parquet")

## 2. Creating a DataSet (domain object)

You create **instances of the generated Pydantic models** like any other Python class. Here we define a single dataset that will group our cells/items.

In [42]:
ds = DataSet(
    id="minnie65_v1412_csm_cluster",
    name="Minnie65 v1412 CSM Dendrite Ultrastructure Collection",
    publication="none",
    modality="ELECTRON_MICROSCOPY",
    project_id="minnie65",
)
ds

DataSet(project_id='minnie65', id='minnie65_v1412_csm_cluster', name='Minnie65 v1412 CSM Dendrite Ultrastructure Collection', publication='none', modality='ELECTRON_MICROSCOPY')

## 3. From Pydantic models to Arrow tables

Three steps are used everywhere in this workflow:

1. **`build_arrow_schema(SomeModel)`** — Builds a PyArrow schema from the Pydantic model’s field types (so column types are consistent).
2. **`models_to_table(instances, schema)`** — Converts a list of model instances into a single PyArrow table (no JSON round-trip).
3. **`attach_linkml_metadata(table, linkml_class="...")`** — Attaches schema/class metadata to the table for provenance (e.g. which LinkML class and version).

In [43]:
schema = build_arrow_schema(DataSet)
table = models_to_table([ds], schema)
table = attach_linkml_metadata(table, linkml_class="DataSet")
table

pyarrow.Table
project_id: string not null
id: string not null
name: string not null
publication: string
modality: string
----
project_id: [["minnie65"]]
id: [["minnie65_v1412_csm_cluster"]]
name: [["Minnie65 v1412 CSM Dendrite Ultrastructure Collection"]]
publication: [["none"]]
modality: [["ELECTRON_MICROSCOPY"]]

## 4. Writing to Delta Lake

Tables are written with **`write_deltalake()`**. Using `partition_by` (e.g. `project_id`) keeps the layout consistent and makes filtering by project efficient. Use a path where you want the Delta table to live (e.g. under `../results/`).

In [44]:
# Uncomment and set PATH to a directory where you want the Delta table (e.g. ../results/dataset/)
PATH = "../results/minnie/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

## 5. DataItems and DataItem–DataSet associations

- **DataItem**: Represents a single “thing” in the project (e.g. a nucleus/cell), with `id`, `name`, `project_id`.
- **DataItemDataSetAssociation**: Links each DataItem to a DataSet (many-to-many). So you can have multiple datasets and assign each cell to one or more of them.

Here: DataItems from **nuc_df**; associations from **dfm** (only cells with features linked to this dataset).

In [45]:
# Build one DataItem per nucleus in nuc_df (already loaded in 2a)
data_items = []
for k, row in nuc_df.iterrows():
    data_items.append(DataItem(
        id=str(row.id),
        name=str(row.pt_root_id),
        project_id="minnie65"))

In [46]:
nuc_df.head()

,id,volume,pt_root_id,orig_root_id,pt_supervoxel_id,pt_position,pt_position_lookup
1,373879,229.045043,864691136090135607,864691136090135607,96218056992431305,"[228816, 239776, 19593]","[228816, 239776, 19593]"
3,201858,93.753836,864691135373893678,864691135373893678,84955554103121097,"[146848, 213600, 26267]","[146848, 213600, 26267]"
4,600774,135.189791,864691135682378744,0,111493022281121981,"[339120, 276112, 19442]","[339520, 276480, 19506]"
5,408486,103.686144,864691135194387242,864691135194387242,98470475952865044,"[245024, 244416, 25074]","[245024, 244416, 25074]"
7,598774,31.230034,864691135741608653,864691135741608653,110718553912730154,"[334096, 273472, 20713]","[334064, 273328, 20701]"


In [47]:
nuc_df.attrs

{'datastack_name': 'minnie65_phase3_v1',
 'join_query': False,
 'table_live_compatible': True,
 'table_voxel_resolution_z': 40.0,
 'table_voxel_resolution_x': 4.0,
 'table_description': 'A table that merges the nucleus_detection_v0 table with the nucleus_alternative_points table to provide one root_id column which can be used to lookup segments associated with root_ids, while preserving the geometric center of the nucleus. ',
 'table_id': 3,
 'table_notice_text': None,
 'table_voxel_resolution_y': 4.0,
 'table_datastack_name': 'minnie65_phase3_v1',
 'table_name': 'nucleus_detection_lookup_v1',
 'dataframe_resolution': [4.0, 4.0, 40.0],
 'filters': {'inclusive': None,
  'exclusive': None,
  'equal': None,
  'greater': None,
  'less': None,
  'greater_equal': None,
  'less_equal': None,
  'spatial': None,
  'regex': None},
 'select_columns': None,
 'offset': None,
 'limit': None,
 'live_query': False,
 'timestamp': None,
 'materialization_version': 1412,
 'column_names': "{'nucleus_detec

In [48]:
schema = build_arrow_schema(DataItem)
table = models_to_table(data_items, schema)
table = attach_linkml_metadata(table, linkml_class="DataItem")
# write_deltalake("../results/dataitem/", table, mode="append", partition_by=["project_id"])
table

pyarrow.Table
project_id: string not null
id: string not null
name: string not null
neuroglancer_link: string
----
project_id: [["minnie65","minnie65","minnie65","minnie65","minnie65",...,"minnie65","minnie65","minnie65","minnie65","minnie65"]]
id: [["373879","201858","600774","408486","598774",...,"232979","598753","111162","528334","267033"]]
name: [["864691136090135607","864691135373893678","864691135682378744","864691135194387242","864691135741608653",...,"864691135496010384","864691135743752909","864691134912248365","864691135968943973","864691135489514810"]]
neuroglancer_link: [[null,null,null,null,null,...,null,null,null,null,null]]

In [49]:
# Link each cell in the feature table (dfm) to this DataSet — only cells with features belong to it
data_item_associations = [
    DataItemDataSetAssociation(dataitem_id=str(row.id), dataset_id=ds.id, project_id="minnie65")
    for k, row in dfm.iterrows()
]

schema = build_arrow_schema(DataItemDataSetAssociation)
table = models_to_table(data_item_associations, schema)
table = attach_linkml_metadata(table, linkml_class="DataItemDataSetAssociation")
table

pyarrow.Table
project_id: string not null
dataitem_id: string not null
dataset_id: string not null
----
project_id: [["minnie65","minnie65","minnie65","minnie65","minnie65",...,"minnie65","minnie65","minnie65","minnie65","minnie65"]]
dataitem_id: [["485509","263203","456177","461339","302377",...,"256280","258113","258355","256602","260817"]]
dataset_id: [["minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster",...,"minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster","minnie65_v1412_csm_cluster"]]

## 6. Querying Delta tables (Polars)

Once data is written to Delta, you **read it with Polars** (or Pandas) and join tables as needed. Typical pattern: filter associations by `project_id` and `dataset_id`, then join to the DataItem table on `dataitem_id` = `id` to get the full item metadata for that dataset.

In [50]:
import polars as pl

# If you wrote Delta tables to ../results/, you can run:
# assoc_df = pl.read_delta("../results/dataitem_dataset_association/")
# items_df = pl.read_delta("../results/dataitem/")
# result = (
#     assoc_df
#     .filter(pl.col("project_id") == "minnie65")
#     .filter(pl.col("dataset_id") == ds.id)
#     .join(items_df, left_on="dataitem_id", right_on="id", how="inner")
# )
# print(result)

# Here we simulate the same join on the in-memory tables we just built
assoc_df = pl.from_arrow(models_to_table(data_item_associations, build_arrow_schema(DataItemDataSetAssociation)))
items_df = pl.from_arrow(models_to_table(data_items, build_arrow_schema(DataItem)))
result = assoc_df.join(items_df, left_on="dataitem_id", right_on="id", how="inner")
result

project_id,dataitem_id,dataset_id,project_id_right,name,neuroglancer_link
str,str,str,str,str,str
"""minnie65""","""373879""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691136090135607""",null
"""minnie65""","""372421""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691136662334942""",null
"""minnie65""","""369908""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691136145011252""",null
"""minnie65""","""406127""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691135741073771""",null
"""minnie65""","""373378""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691136923735524""",null
…,…,…,…,…,…
"""minnie65""","""338345""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691135837709971""",null
"""minnie65""","""303490""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691135476591168""",null
"""minnie65""","""233088""","""minnie65_v1412_csm_cluster""","""minnie65""","""864691135430122800""",null


## 7. Cell features: definitions and feature sets

- **CellFeatureDefinition**: Describes one named feature (id, description, unit, data_type, optional range_min/range_max). The original notebook loads these from a CSV (e.g. `minnie_cell_features.csv`).
- **CellFeatureSet**: Groups a list of feature definition IDs and describes the set (id, description, feature_definition_ids, extraction_method). One feature set can be used for many cells.

In [51]:
units_df = pd.read_csv('../data/minnie1412/minnie_cell_features.csv')
units_df

,id,description,unit,data_type,range_min,range_max
0,nucleus_volume_um,Nucleus volume,MICRONS_CUBED,<f4,0.0,NaN
1,nucleus_area_um,Nucleus surface area,MICRONS_SQUARE,<f4,0.0,NaN
2,nuclear_area_to_volume_ratio,Nucleus surface area to volume ratio,MICRONS_INVERSE,<f4,0.0,NaN
3,nuclear_folding_area_um,Area of nucleus in an infolding (see Elabbady ...,MICRONS_SQUARE,<f4,0.0,NaN
4,fraction_nuclear_folding,Fraction of nucleus in an infolding,RATIO,<f4,0.0,1.0
...,...,...,...,...,...,...
77,branch_svd3,SVD loading dendritic path length vs distance ...,RATIO,<f4,NaN,NaN
78,branch_svd4,SVD loading dendritic path length vs distance ...,RATIO,<f4,NaN,NaN
79,ego_count_pca0,PC loading synapse depth relative to soma comp...,RATIO,<f4,NaN,NaN
80,ego_count_pca1,PC loading synapse depth relative to soma comp...,RATIO,<f4,NaN,NaN


In [52]:
fds=[]
for idx, row in units_df.iterrows():
    fd=CellFeatureDefinition(
        id=str(row['id']),
        description=str(row['description']),
        unit=str(row['unit']),
        data_type=str(row['data_type']),
        range_min=float(row['range_min']) if pd.notna(row['range_min']) else None,
        range_max=float(row['range_max']) if pd.notna(row['range_max']) else None
    )
    fds.append(fd)

In [53]:
schema_fd = build_arrow_schema(CellFeatureDefinition)
table_fd = models_to_table(fds, schema_fd)
table_fd = attach_linkml_metadata(table_fd, linkml_class="CellFeatureDefinition")

In [54]:
cfs = CellFeatureSet(
    id='csm_cluster_features',
    description='Cell features used for clustering in the Allen Institute\'s large scale EM projects. ' \
                'Contains features from Elabbady et al 2025, Scheider-Mizell et al 2025, and ' \
                'some more recent features dervied from spine detection from Ben Pedigo. '
                'Feature set developed by Casey Schneider-Mizell.  ' \
                'Tries to take a synapse centric morphological approach with features ' \
                'describing how synapse densities are distributed across the dendritic arbors.',
    feature_definition_ids=[fd.id for fd in fds],
    extraction_method='Aggegated and computed via https://github.com/AllenInstitute/em_skeleton_feature_extraction.'
)
    

In [55]:
schema_cfs = build_arrow_schema(CellFeatureSet)
table_cfs = models_to_table([cfs], schema_cfs)
table_cfs = attach_linkml_metadata(table_cfs, linkml_class="CellFeatureSet")
table_cfs

pyarrow.Table
id: string not null
description: string
feature_definition_ids: list<item: string>
  child 0, item: string
extraction_method: string
----
id: [["csm_cluster_features"]]
description: [["Cell features used for clustering in the Allen Institute's large scale EM projects. Contains featu (... 330 chars omitted)"]]
feature_definition_ids: [[["nucleus_volume_um","nucleus_area_um","nuclear_area_to_volume_ratio","nuclear_folding_area_um","fraction_nuclear_folding",...,"branch_svd3","branch_svd4","ego_count_pca0","ego_count_pca1","ego_count_pca2"]]]
extraction_method: [["Aggegated and computed via https://github.com/AllenInstitute/em_skeleton_feature_extraction."]]

## 8. Cell feature matrix (per-cell feature values)

Feature **definitions** describe what each column means; the **feature matrix** is the table of actual values: one row per cell (e.g. `id`), one column per feature, plus `project_id` and `feature_set_id`. **`build_cell_feature_matrix_schema(feature_set, list_of_definitions, cell_index_column="id")`** returns a PyArrow schema that matches this layout. You then build a DataFrame with the same columns and cast types to match the definitions (e.g. float32 for `<f4`), and convert to a PyArrow table with that schema before writing to Delta.

### 8b. Cortical coordinates (Minnie standard_transform)

The **standard_transform** package converts voxel positions to a leveled cortical coordinate system (y=0 at pia). We apply it to `nuc_df` to get x (medial–lateral), y (dorsal–ventral), z (caudal–rostral) in mm, then define a small feature set `minnie65_std_transform_coordinates` and write it as a cell feature matrix.

In [56]:
tform_nm = standard_transform.minnie_transform_vx()
xt = tform_nm.apply_dataframe("pt_position", nuc_df, projection="x")
yt = tform_nm.apply_dataframe("pt_position", nuc_df, projection="y")
zt = tform_nm.apply_dataframe("pt_position", nuc_df, projection="z")

cortical_coord_df = pd.DataFrame()
cortical_coord_df["id"] = nuc_df.id.astype(str)
cortical_coord_df["x_medial-lateral"] = np.array(xt, dtype=np.float32) / 1000
cortical_coord_df["y_dorsal-ventral"] = np.array(yt, dtype=np.float32) / 1000
cortical_coord_df["z_caudal-rostral"] = np.array(zt, dtype=np.float32) / 1000

cfd_x = CellFeatureDefinition(
    id="x_medial-lateral",
    description="x in minnie65 after standard_transform (medial–lateral).",
    unit=Unit.MICRONS_LENGTH,
    data_type="<f4",
)
cfd_y = CellFeatureDefinition(
    id="y_dorsal-ventral",
    description="y in minnie65 after standard_transform (dorsal–ventral, 0 at pia).",
    unit=Unit.MICRONS_LENGTH,
    data_type="<f4",
)
cfd_z = CellFeatureDefinition(
    id="z_caudal-rostral",
    description="z in minnie65 after standard_transform (caudal–rostral).",
    unit=Unit.MICRONS_LENGTH,
    data_type="<f4",
)
cfs_microns_stdtform = CellFeatureSet(
    id="minnie65_std_transform_coordinates",
    description="Soma coordinates after standard_transform (y=0 at pia).",
    feature_definition_ids=[cfd_x.id, cfd_y.id, cfd_z.id],
    extraction_method="standard_transform on nucleus_detection_lookup_v1, pt_root_id!=0.",
)

In [57]:
# Write coordinate feature definitions and set, then the coordinate feature matrix
schema_fd = build_arrow_schema(CellFeatureDefinition)
table_fd = models_to_table([cfd_x, cfd_y, cfd_z], schema_fd)
table_fd = attach_linkml_metadata(table_fd, linkml_class="CellFeatureDefinition")
# write_deltalake("../results/cellfeaturedefinition/", table_fd, mode="append")

schema_cfs = build_arrow_schema(CellFeatureSet)
table_cfs = models_to_table([cfs_microns_stdtform], schema_cfs)
table_cfs = attach_linkml_metadata(table_cfs, linkml_class="CellFeatureSet")
# write_deltalake("../results/cellfeatureset/", table_cfs, mode="append")

schema_coord = build_cell_feature_matrix_schema(cfs_microns_stdtform, [cfd_x, cfd_y, cfd_z], cell_index_column="id")
cortical_coord_df["project_id"] = "minnie65"
cortical_coord_df["feature_set_id"] = "minnie65_std_transform_coordinates"
table_coord = pa.Table.from_pandas(cortical_coord_df, schema=schema_coord, preserve_index=False)
# write_deltalake("../results/cellfeatures/minnie65_std_transform_coordinates/", table_coord, mode="append", partition_by=["project_id", "feature_set_id"])
table_coord

pyarrow.Table
id: string not null
project_id: string not null
feature_set_id: string not null
x_medial-lateral: float
y_dorsal-ventral: float
z_caudal-rostral: float
----
id: [["373879","201858","600774","408486","598774",...,"232979","598753","111162","528334","267033"]]
project_id: [["minnie65","minnie65","minnie65","minnie65","minnie65",...,"minnie65","minnie65","minnie65","minnie65","minnie65"]]
feature_set_id: [["minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates",...,"minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates","minnie65_std_transform_coordinates"]]
x_medial-lateral: [[0.8281897,0.5106909,1.2550592,0.8911574,1.2359601,...,0.51900256,1.231387,0.3743121,1.1189463,0.6236822]]
y_dorsal-ventral: [[0.6385538,0.5056723,0.8217993,0.6626937,0.80952793,...,0.

In [58]:
# Build schema and feature matrix from Minnie feature table (dfm)
schema = build_cell_feature_matrix_schema(cfs, fds, cell_index_column="id")

drop_list = [c for c in ['root_id', 'pt_root_id_y', 'valence'] if c in dfm.columns]
df = dfm.drop(columns=drop_list, axis=1)
df['project_id'] = 'minnie65'
df['feature_set_id'] = 'csm_cluster_features'
df['id'] = dfm['id'].astype('string')
for cfd in fds:
    col = cfd.id
    if col in df.columns and cfd.data_type:
        if cfd.data_type[1] == 'f':
            df[col] = df[col].astype('float32')
        elif cfd.data_type[1] == 'i':
            df[col] = df[col].astype('int32')

table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
table

pyarrow.Table
id: string not null
project_id: string not null
feature_set_id: string not null
nucleus_volume_um: float
nucleus_area_um: float
nuclear_area_to_volume_ratio: float
nuclear_folding_area_um: float
fraction_nuclear_folding: float
nucleus_to_soma_ratio: float
soma_volume_um: float
soma_area_um: float
soma_to_nucleus_center_dist: float
soma_area_to_volume_ratio: float
soma_synapse_density_um: float
tip_len_dist_dendrite_p75: float
tip_tort_dendrite_p75: float
num_syn_dendrite: int32
num_syn_soma: int32
path_length_dendrite: float
radial_extent_dendrite: float
syn_dist_distribution_dendrite_p50: float
syn_size_distribution_soma_p50: int32
syn_size_distribution_dendrite_p50: int32
syn_size_distribution_dendrite_p5: int32
syn_size_distribution_dendrite_p95: int32
syn_size_dendrite_cv: float
syn_depth_dist_p1: float
syn_depth_dist_p99: float
syn_depth_extent: float
median_density: float
radius_dist: float
area_factor: float
dendrite_length_binned_0: float
dendrite_length_binned_1:

## 9. Clusters and ClusterMembership

- **Cluster**: A node in a taxonomy (e.g. neuron → glutamatergic / gabaergic → L4IT, PTC, …). Fields include `id`, `parent`, `children`, `level`, `hex_color`, `heirachy_category`, `project_id`.
- **ClusterMembership**: Assigns each “item” (e.g. cell id) to a cluster, optionally with `probability`. The original notebook builds clusters from CAVE’s `cell_type_multifeature_v1` and then one membership row per (cell, cluster) at each level (neuron, class, subtype).

## 10. Cell–cell connectivity (long format)

**CellCellConnectivityLong** stores one row per (presynaptic_cell, postsynaptic_cell, measurement_type): e.g. synapse count or sum anatomical size. Each row has `id`, `presynaptic_cell`, `postsynaptic_cell`, `measurement_type` (enum, e.g. `SYNAPSE_COUNT`, `SUM_ANATOMICAL_SIZE`), `value`, `unit`, `project_id`. The original notebook reads a connectivity parquet, filters by proofread axons, then builds two Delta tables—one for synapse count and one for sum size—partitioned by `project_id` and `measurement_type`.

In [72]:
# Minnie: soma–soma connectivity parquet, filtered to proofread axons only; then SYNAPSE_COUNT and SUM_ANATOMICAL_SIZE
conn_df = pd.read_parquet('../data/minnie1412/minnie_soma_soma_connectivity.parquet')
prf_df = client.materialize.query_table('proofreading_status_and_strategy', materialization_version=1412)
prf_axons = prf_df.query('status_axon')
conn_df = conn_df[conn_df.pre_pt_root_id.isin(prf_axons.pt_root_id)]

cccls = []
for k, row in conn_df.iterrows():
    cccls.append(CellCellConnectivityLong(id=str(k), presynaptic_cell=str(row.pre_nuc_id), postsynaptic_cell=str(row.post_nuc_id), measurement_type=SynapticMeasurementType.SYNAPSE_COUNT, value=row.n_syn, unit=Unit.COUNT, project_id='minnie65'))
schema = build_arrow_schema(CellCellConnectivityLong)
table = models_to_table(cccls, schema)
table = attach_linkml_metadata(table, linkml_class="CellCellConnectivityLong")
# write_deltalake("../results/cellcellconnectivitylong/", table, mode="append", partition_by=["project_id", "measurement_type"])
table

# Same connectivity in long form for SUM_ANATOMICAL_SIZE (sum_size)
cccls_size = [CellCellConnectivityLong(id=str(k), presynaptic_cell=str(row.pre_nuc_id), postsynaptic_cell=str(row.post_nuc_id), measurement_type=SynapticMeasurementType.SUM_ANATOMICAL_SIZE, value=row.sum_size, unit=Unit.ARBITRARY_UNIT, project_id='minnie65') for k, row in conn_df.iterrows()]
table_size = models_to_table(cccls_size, schema)
table_size = attach_linkml_metadata(table_size, linkml_class="CellCellConnectivityLong")
# write_deltalake("../results/cellcellconnectivitylong/", table_size, mode="append", partition_by=["project_id", "measurement_type"])
table_size

pyarrow.Table
project_id: string not null
id: string not null
description: string
presynaptic_cell: string
postsynaptic_cell: string
measurement_type: string
modality: string
value: double not null
unit: string not null
----
project_id: [["minnie65","minnie65","minnie65","minnie65","minnie65",...,"minnie65","minnie65","minnie65","minnie65","minnie65"]]
id: [["43176584","43181492","43181499","43183398","43183978",...,"303199984","303200956","303201438","303201480","303202316"]]
description: [[null,null,null,null,null,...,null,null,null,null,null]]
presynaptic_cell: [["337175","330167","273595","301121","273595",...,"497103","463716","490761","363525","520364"]]
postsynaptic_cell: [["304043","339142","339142","339142","205051",...,"422139","422139","422139","422139","422139"]]
measurement_type: [["SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE",...,"SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE","SUM_ANATOMICAL_SIZE","SUM_ANATOMICA

---

## Summary

| Step | What it does |
|------|----------------|
| **LinkML YAML** | Defines classes and slots in `schemas/`. |
| **generate_models** | Produces Pydantic models in `models.py`. |
| **Your code** | Instantiates models (DataSet, DataItem, Cluster, …) from your data sources. |
| **build_arrow_schema(Model)** | Gets a PyArrow schema for that model. |
| **models_to_table(instances, schema)** | Converts model instances to one PyArrow table. |
| **attach_linkml_metadata(table, linkml_class=...)** | Adds schema/version metadata. |
| **write_deltalake(path, table, partition_by=...)** | Writes the table to Delta Lake. |
| **Polars/Pandas** | Read Delta and join/filter for downstream analysis. |

The full `parse_minnie_clustering.ipynb` does the same pattern at scale: CAVE/materialization tables and parquet inputs → model instances → Arrow → Delta; plus coordinate transforms, UMAP, and proofread-only datasets. This notebook keeps only the **reusable workflow** and explains each part.